In [0]:
import requests
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

# -----------------------------
# CONFIG
# -----------------------------
PRODUCTS = [
    "BTC-USD",
    "ETH-USD",
    "SOL-USD",
    "XRP-USD",
    "DOGE-USD",
    "ADA-USD",
    "AVAX-USD",
    "LINK-USD",
    "LTC-USD",
    "BCH-USD",
    "AAVE-USD",
    "DOT-USD",
    "UNI-USD",
    "ATOM-USD",
    "NEAR-USD"
]

START_DATE = datetime(2020, 1, 1, tzinfo=timezone.utc)
END_DATE = datetime.now(timezone.utc)

BASE_URL = "https://api.coinbase.com/api/v3/brokerage/market/products"

# -----------------------------
# FUNCTION TO GET ONE WINDOW
# -----------------------------
def get_coinbase_candles(product_id, start_dt, end_dt):

    url = f"{BASE_URL}/{product_id}/candles"

    params = {
        "start": str(int(start_dt.timestamp())),
        "end": str(int(end_dt.timestamp())),
        "granularity": "ONE_DAY",
        "limit": 350
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    candles = response.json()["candles"]

    rows = []

    for c in candles:
        rows.append({
            "product_id": product_id,
            "timestamp": int(c["start"]),
            "open": float(c["open"]),
            "high": float(c["high"]),
            "low": float(c["low"]),
            "close": float(c["close"]),
            "volume": float(c["volume"])
        })

    return rows

# -----------------------------
# PULL ALL DATA
# -----------------------------
all_rows = []

for product in PRODUCTS:

    current_start = START_DATE

    while current_start < END_DATE:

        current_end = min(
            current_start + timedelta(days=300),
            END_DATE
        )

        print(
            f"Pulling {product}: "
            f"{current_start.date()} to {current_end.date()}"
        )

        try:
            rows = get_coinbase_candles(
                product,
                current_start,
                current_end
            )

            all_rows.extend(rows)

        except Exception as e:
            print(f"Error pulling {product}: {e}")

        current_start = current_end

# -----------------------------
# CREATE SPARK DATAFRAME
# -----------------------------
crypto_df = spark.createDataFrame(all_rows)

crypto_df = (
    crypto_df
    .withColumn(
        "trade_timestamp",
        F.from_unixtime("timestamp").cast("timestamp")
    )
    .withColumn(
        "trade_date",
        F.to_date("trade_timestamp")
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source",
        F.lit("Coinbase")
    )
)

# Remove duplicate candles just in case date windows overlap
crypto_df = crypto_df.dropDuplicates(
    ["product_id", "timestamp"]
)

# -----------------------------
# CREATE BRONZE SCHEMA
# -----------------------------
spark.sql("""
CREATE SCHEMA IF NOT EXISTS bronze
""")

# -----------------------------
# SAVE AS DELTA TABLE
# -----------------------------
(
    crypto_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.coinbase_crypto_candles_raw")
)

# -----------------------------
# RESULTS
# -----------------------------
print(f"Total rows loaded: {crypto_df.count():,}")

print("Rows by product:")
crypto_df.groupBy("product_id") \
    .count() \
    .orderBy("product_id") \
    .show()

display(
    crypto_df
    .select(
        "product_id",
        "trade_date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "_ingested_at"
    )
    .orderBy("product_id", "trade_date")
)